# Setup do Ambiente S3 — Judicial Analytics Platform

Este notebook configura **todo o ambiente do zero**. Deve ser rodado uma única vez por workspace.
Se o workspace for recriado ou resetado, rode este notebook antes de qualquer outro.

---

## Sobre o projeto

A **Judicial Analytics Platform** é uma plataforma analítica operacional do Judiciário brasileiro
construída sobre dados públicos do CNJ (DataJud + Justiça em Números).

A arquitetura separa deliberadamente **compute** e **storage**:

- **Compute → Databricks** — notebooks, SQL Warehouse, Jobs, Unity Catalog
- **Storage → AWS S3** — arquivos Delta Lake (`.parquet` + `_delta_log/`) ficam no bucket S3, não no DBFS

---

## Pré-requisito:

Este notebook assume que o **Setup AWS + Databricks** já foi concluído. Isso significa:

- Bucket S3 `judicial-analytics-storage` criado na AWS
- IAM Role com cross-account trust configurado via AWS CloudFormation (Quickstart do Databricks)
- Storage Credential e External Location criados no Unity Catalog apontando para o bucket
- Teste de conexão passou no Unity Catalog

---

## Estado que este notebook reproduz

```
judicial/                          ← catálogo → aponta para s3://judicial-analytics-storage/
├── bronze/                        ← schema: dados brutos da API
│   ├── Volumes/raw_files/         ← arquivos JSON antes de virar tabela Delta
│   │   ├── TJSC/
│   │   ├── TJPR/
│   │   └── processados/TJSC/...
│   └── Tables/                    ← criadas pelos notebooks de ingestão
├── silver/                        ← schema: dados limpos e transformados
└── gold/                          ← schema: modelo analítico final
```

## 1. Verificar ambiente

In [0]:
# Confirmar que o Spark está funcionando antes de qualquer coisa
print(f"Spark version: {spark.version}")

Spark version: 4.1.0


## 2. Criar Catalog

In [0]:
%sql
-- Catalog raiz do projeto
-- Agrupa todos os schemas: bronze, silver, gold

CREATE CATALOG IF NOT EXISTS judicial
  MANAGED LOCATION 's3://judicial-analytics-storage';

In [0]:
%sql
DESCRIBE CATALOG EXTENDED judicial;

info_name,info_value
Catalog Name,judicial
Comment,
Owner,viniciusgzanatta@outlook.com
Catalog Type,Regular
Created By,viniciusgzanatta@outlook.com
Created At,2026-06-02 AD at 18:33:57 UTC
Updated By,viniciusgzanatta@outlook.com
Updated At,2026-06-02 AD at 20:33:01 UTC
Storage Root,s3://judicial-analytics-storage/
Storage Location,s3://judicial-analytics-storage/__unitystorage/catalogs/1be6c2be-d4ee-489f-aa76-ac5a57a442c1


## 3. Criar Schemas (camadas medalhão)

In [0]:
%sql
-- bronze: dados brutos da API, sem transformação
CREATE SCHEMA IF NOT EXISTS judicial.bronze

In [0]:
%sql
-- silver: dados limpos, deduplicados e enriquecidos
CREATE SCHEMA IF NOT EXISTS judicial.silver

In [0]:
%sql
-- gold: modelo analítico final (dimensões e fatos)
CREATE SCHEMA IF NOT EXISTS judicial.gold

## 4. Criar Volume de arquivos brutos

O Volume `raw_files` é onde os arquivos JSON gerados são armazenados
antes de passarem pela ingestão do Delta Lake propriamente.

Aqui tive muitos problemas com relação ao dbutils e outras funcionalidades(dbutils aparentemente só funciona para ambientes DBFS(Data Bricks File System)), mas será criado um **EXTERNAL VOLUME**, que quer dizer que o volume **não vai ser gerenciado pelo Unity Catalog**.

In [0]:
%sql
CREATE EXTERNAL VOLUME judicial.bronze.raw_files
  LOCATION 's3://judicial-analytics-storage/bronze/raw_files';



In [0]:
%sql
--Verificar a existência do volume externo criado
SELECT 
    volume_catalog,
    volume_schema,
    volume_name,
    volume_type,
    storage_location,
    comment
FROM system.information_schema.volumes
WHERE volume_type = 'EXTERNAL';

volume_catalog,volume_schema,volume_name,volume_type,storage_location,comment
judicial,bronze,raw_files,EXTERNAL,s3://judicial-analytics-storage/bronze/raw_files,null


## 5. Criar estrutura de diretórios no Volume

Cada tribunal tem sua própria pasta dentro do Volume.
A pasta `processados/` recebe os arquivos após serem ingeridos na Bronze
— evita que o mesmo arquivo seja lido duas vezes em caso de reexecução.

In [0]:
%python
import os

VOLUME_BASE = "/Volumes/judicial/bronze/raw_files"
TRIBUNAIS   = ["TJRS", "TJPR", "TJSC", "TJSP"]

for tribunal in TRIBUNAIS:
    os.makedirs(f"{VOLUME_BASE}/{tribunal}", exist_ok=True)
    print(f"Criado: {VOLUME_BASE}/{tribunal}/")

for tribunal in TRIBUNAIS:
    os.makedirs(f"{VOLUME_BASE}/processados/{tribunal}", exist_ok=True)
    print(f"Criado: {VOLUME_BASE}/processados/{tribunal}/")

Criado: /Volumes/judicial/bronze/raw_files/TJRS/
Criado: /Volumes/judicial/bronze/raw_files/TJPR/
Criado: /Volumes/judicial/bronze/raw_files/TJSC/
Criado: /Volumes/judicial/bronze/raw_files/TJSP/
Criado: /Volumes/judicial/bronze/raw_files/processados/TJRS/
Criado: /Volumes/judicial/bronze/raw_files/processados/TJPR/
Criado: /Volumes/judicial/bronze/raw_files/processados/TJSC/
Criado: /Volumes/judicial/bronze/raw_files/processados/TJSP/


## 6. Verificação final

Rode todas as células abaixo e confirme que a saída bate com o esperado antes de avançar.

In [0]:
%sql
-- Esperado: bronze, default, gold, information_schema, silver
SHOW SCHEMAS IN judicial

databaseName
bronze
default
gold
information_schema
silver


In [0]:
%sql
-- Esperado: raw_files
SHOW VOLUMES IN judicial.bronze

database,volume_name
bronze,raw_files


In [0]:
# Esperado: rawfiles: ['TJPR', 'TJRS', 'TJSC', 'TJSP'], 'processados' Processados: ['TJPR', 'TJRS', 'TJSC', 'TJSP']
import os
VOLUME_BASE = "/Volumes/judicial/bronze/raw_files"

print(str(os.listdir(VOLUME_BASE)) + str(os.listdir(VOLUME_BASE+"/processados")))

['TJPR', 'TJRS', 'TJSC', 'TJSP', 'processados']['TJPR', 'TJRS', 'TJSC', 'TJSP']
